---
title: 1a) The Connectome Interpreter Tutorial - Linear Methods (Effective Connectivity)
authors:
  - name: Aarushi Vardhan
    affiliations:
      - University of Toronto / University of Cambridge
  - name: Sapolnach Prompiengchai
    affiliations:
      - University of Oxford
---

# Install Connectome Interpreter

Before starting, please install the **Connectome Interpreter** package and download the connectivity data used in this tutorial. Open your terminal and activate the environment you will use for this tutorial. Then run:

```bash
pip install git+https://github.com/YijieYin/connectome_interpreter.git --no-deps
```

This tutorial is adapted from the [Linear Methods tutorial notebook](https://github.com/YijieYin/connectome_interpreter) of The Connectome Interpreter. Please refer to the documentation for [additonal information ](https://connectome-interpreter.readthedocs.io/en/latest/modules/toc.html). 

In [ ]:
# 1. load your packages at the top
import pandas as pd
project_dir = '/Users/aarushivardhan/Desktop/MB'
from connectome_interpreter import *
import scipy as sp


# Get cell types function list. 
sheet_id = "1VHCEnurOdb4FDC_NUKZX_BpBckQ9LpKxv0CsK_ObVok"
url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid=0"
df = pd.read_csv(url)
df = df.rename(columns={'función conocida': 'known_function'}) # run this only if the column is called "función conocida", otherwise leave it out
type_to_function = dict(zip(df["cell_type"], df["known_function"])) # type_to_function contains a list of cell types and their experimentally-tested functions. 

# What is effective connectivity?

So far, we have looked at **direct connections** between neurons. For example, suppose neuron A connects directly to neuron C:

$$
A \longrightarrow C
$$

We can describe the strength of this connection as the fraction of C's input that comes from A. For example, if A provides 60% of C's input, the connection strength is

$$
A\rightarrow C = 0.60.
$$

We looked at this type of normalisation in [Notebook 1C of the neuPrint tutorial](./6c-neuprint_plotting.ipynb).

But neurons do not only influence neurons to which they are directly connected. Activity can also travel through **intermediate neurons**.

For example:

$$
A \longrightarrow B \longrightarrow C
$$

Here, A does not connect directly to C. Nevertheless, A can influence C through B.

This is the basic idea behind **effective connectivity**: we want to measure how strongly one neuron can influence another through indirect, multi-step paths.

## Following one indirect path

Consider the following pathway:

$$
A \xrightarrow{40\%} B \xrightarrow{60\%} C
$$

The percentages tell us how much of the **target neuron's input** comes from the source neuron.

So:

* A provides 40% of B's input.
* B provides 60% of C's input.

We want to know:

> How much of C's input can be attributed to A through the path {math}`A \rightarrow B \rightarrow C`?

We multiply the two fractions:

$$
0.40\times0.60=0.24.
$$

So the effective connection from A to C **along this path** is

$$
A\rightarrow B\rightarrow C = 0.24.
$$

In other words, A accounts for 24% of C's input through this particular two-step path.

### Why do we multiply?

Imagine C receives 100 units of input.

B provides 60% of C's input:

$$
100\times0.60=60.
$$

So 60 of those units come through B.

But only 40% of B's input came from A. Therefore, the amount attributable to A is

$$
60\times0.40=24.
$$

Thus,

$$
\frac{24}{100}=0.24.
$$

This gives us a useful rule:

$$
\boxed{\text{For connections along a path, multiply the connection strengths.}}
$$

## Representing direct connections with a matrix

<div style="text-align: center;">

:::{figure} ../static/effective_connectivity.png
:width: 1000px

**Effective connectivity Schematic.** A) Is a toy circuit with neurons a, b, c (left) and its representaiton in matrix form (right). B) is showing the matrix multiplication from neuron a to a. Adapted from [Yijie Yin, 2024](https://connectome-interpreter.readthedocs.io/en/latest/tutorials/matmul.html).
:::

</div>

Doing these calculations one neuron at a time would quickly become impractical. Instead, we can represent all the direct connections in a **matrix**.

For the toy circuit in the figure above (A, left), the connectivity matrix is shown (B, right)

The **rows** represent the source, or presynaptic, neurons:

$$
\text{row}=\text{{pre}}
$$

and the **columns** represent the target, or postsynaptic, neurons:

$$
\text{column}=\text{{post}}.
$$
Therefore, an entry {math}`M_{ij}` means

$$
\boxed{M_{ij}=\text{direct connection from neuron }i\text{ to neuron }j.}
$$

For example,

$$
M_{ab}=0.3
$$

in the matrix means:

$$
a\rightarrow b=0.3.
$$

The first row of the matrix is

$$
[0,\;0.3,\;0.5].
$$

This tells us all the direct connections leaving neuron \(a\):

$$
a\rightarrow a=0,
\qquad
a\rightarrow b=0.3,
\qquad
a\rightarrow c=0.5.
$$

Notice that this matrix contains only **direct, one-hop connections**.

So:

$$
\boxed{M=\text{connectivity after exactly one hop}}
$$

## From one hop to two hops

What if we want to know how strongly \(a\) can influence another neuron after going through **one intermediate neuron**?

That requires two connections (two arrows on the toy circuit), or **two hops**:

$$
a\rightarrow ?\rightarrow \text{target}.
$$

Let's say I want to know how strongly \(a\) can reach \(a\) in exactly two hops?

There are three possible intermediate neurons:
$$
a\rightarrow a\rightarrow a,
\qquad
a\rightarrow b\rightarrow a,
\qquad
a\rightarrow c\rightarrow a
$$

We calculate the strength of each path separately.

Through \(a\):

$$
a\xrightarrow{0}a\xrightarrow{0}a
$$

so

$$
0 \times 0 = 0.
$$

You might be wondering: how can {math}`a \rightarrow a \rightarrow a` connectivity exist? Remember, theoretically, our toy circuit could have self-connectivity. See the grey dashed lines:


<div style="text-align: center;">

:::{figure} ../static/effective_connectivity_annotate.png
:width: 1000px

**Effective connectivity Schematic.** A) Is a toy circuit with neurons a, b, c (left) and its representaiton in matrix form (right). B) is showing the matrix multiplication from neuron a to a. Adapted from [Yijie Yin, 2024](https://connectome-interpreter.readthedocs.io/en/latest/tutorials/matmul.html). The dashed grey lines in A denote hypothetical self-connectivity.
:::

</div>

You may notice that it is possible to have self-connections. Out circuit just doesn't have any! 

Through \(b\):

$$
a\xrightarrow{0.3}b\xrightarrow{0.2}a
$$

so

$$
0.3\times0.2=0.06.
$$

Through \(c\):

$$
a\xrightarrow{0.5}c\xrightarrow{0.8}a
$$

so

$$
0.5\times0.8=0.40.
$$

Sincer there are **more than one possible path** from \(a\) back to \(a\). We therefore add the contributions from all possible paths:

$$
0+0.06+0.40=0.46.
$$

This gives us our second rule:

$$
\boxed{\text{Multiply along a path; add across different paths.}}
$$

## Where does matrix multiplication come in?

Matrix multiplication performs exactly the calculation we just did.

To calculate effective connectivity of \(a,a\), we take **row \(a\)** from the first matrix:

$$
\begin{bmatrix}
0&0.3&0.5
\end{bmatrix}
$$

and **column \(a\)** from the second matrix:

$$
\begin{bmatrix}
0\\
0.2\\
0.8
\end{bmatrix}
$$

We multiply corresponding entries and add them:

$$
(M^2)_{aa}
=
\begin{bmatrix}
0&0.3&0.5
\end{bmatrix}
\begin{bmatrix}
0\\
0.2\\
0.8
\end{bmatrix}
$$

$$
=0(0)+0.3(0.2)+0.5(0.8)
$$

$$
=0+0.06+0.40
$$

$$
=\boxed{0.46}.
$$

Each multiplication corresponds to **one possible path**:

$$
\underbrace{0(0)}_{a\rightarrow a\rightarrow a}
+
\underbrace{0.3(0.2)}_{a\rightarrow b\rightarrow a}
+
\underbrace{0.5(0.8)}_{a\rightarrow c\rightarrow a}.
$$

This is why matrix multiplication is so useful for connectivity analysis. It automatically finds the contribution of every possible intermediate neuron.

For a real connectome, there may be thousands or millions of possible paths. Rather than calculating them individually, matrix multiplication performs all of these calculations simultaneously.

## Why do the columns sum to 1?

There is one final property of this particular connectivity matrix that is important.

Remember that a **column represents a target neuron**.

For example, column \(a\) contains

$$
\begin{bmatrix}
0\\
0.2\\
0.8
\end{bmatrix}
$$

This means that neuron \(a\) receives:

$$
0\% \text{ from }a,
\qquad
20\% \text{ from }b,
\qquad
80\% \text{ from }c
$$

Together,

$$
0+0.2+0.8=1.
$$

Or, in percentages,

$$
0\%+20\%+80\%=100\%.
$$

The same is true for every column:

$$
\boxed{\text{Each column sums to 1 because it represents 100\% of a target neuron's input.}}
$$

This also gives a useful way to remember the orientation of the matrix:

$$
\boxed{\text{rows = where the signal comes from/ source}}
$$

$$
\boxed{\text{columns = where the signal goes to / target}}
$$


# Download the male CNS connectivity data

Choose a directory where you would like to store the data. Run the command below on terminal. Replace `YOUR DIRECTORY` below with that path:

```bash
wget -P 'YOUR DIRECTORY' https://github.com/YijieYin/connectome_data_prep/raw/refs/heads/main/data/maleCNS/mcns_inprop_all_neuron.npz
```

For example:

```bash
wget -P project_dir https://github.com/YijieYin/connectome_data_prep/raw/refs/heads/main/data/maleCNS/mcns_inprop_all_neuron.npz
```

The downloaded file is called `mcns_inprop_all_neuron.npz`.

## What is this file?

This file contains the connectivity matrix for the male CNS. In particular:

* `*_inprop` — input-normalised connectivity. Each entry represents the fraction of a postsynaptic neuron's input that comes from a given presynaptic neuron.

This is the connectivity matrix we will use to calculate **effective connectivity** in this tutorial.


Note: in this tutorial and competition we are focusing on the maleCNS connectome. If you are interested, you are welcome to explore the other connectomes that The Connectome Interpreter supports! 


In [17]:
mcns_inprop = sp.sparse.load_npz(project_dir + '/mcns_inprop_all_neuron.npz')
mcns_inprop

<Compressed Sparse Column sparse matrix of dtype 'float32'
	with 25083972 stored elements and shape (161429, 161429)>

# Wait ... Why can't I see my dataframe?

You may be confused by this output: *Compressed Sparse Column sparse matrix of dtype 'float32' with 25083972 stored elements and shape (161429, 161429)*

Please refer to this section of the [documentation](https://connectome-interpreter.readthedocs.io/en/latest/your_own_data.html#why-does-it-need-to-be-sparse) for a more detailed explanation. 

Essentially, our connectome is mostly sparse (most neurons don't connect to each other). Instead of storing one gigantic memory-intensive connectome matrix (~10GB), we can store this information into something called an edge list (basically only the connections that actually exist are recorded with a pre-synaptic neuron, post-synaptic neuron, and the input proportion columns).

| pre | post | inprop |
| --- | ---- | ------ |
| 101 | 205  |   0.32 |
| 101 | 873  |   0.15 |
| 205 | 992  |   0.71 |
| 873 | 101  |   0.04 |


# What does float32 mean?

A **bit** is the smallest unit of computer information. It can be either:

* `0`
* `1`

So **1 bit = one 0 or 1**.

A **byte** is 8 bits:

`10110101` = 8 bits = 1 byte.

A **float** is a way of storing a number that can have a decimal, such as:

```text
0.5
3.14
0.001
25.75
```

The number after `float` tells you **how many bits the computer uses to store that number**.

For example:

* `float16` → 16 bits → 2 bytes
* `float32` → 32 bits → 4 bytes
* `float64` → 64 bits → 8 bytes

More bits generally means the computer can represent numbers **more precisely**, but it also uses **more memory**.

So:

```python
float32
```

"Store each decimal number using 32 little 0s and 1s (32 bits), which takes 4 bytes of memory."

For this connectome, if you have millions of connection weights, using `float32` instead of `float64` can cut the memory needed for those weights roughly **in half**.



# Loading metadata 

Since the data in our connectivity matrix is only connections between neurons. What if we have a biological question to ask? Like cell types etc. For each neuron (idx) in your matrix you have you have information describing the neurons themselves. This is mcns_meta. 

If you run mcns_meta.[column_of_interest] you can pick a column of interest


In [18]:
# Import metadata
mcns_meta = pd.read_csv(
    'https://raw.githubusercontent.com/YijieYin/connectome_data_prep/refs/heads/main/data/maleCNS/mcns_all_neuron_meta.csv',
    index_col=0,
)
mcns_meta

/var/folders/2f/zb94ck9x7sv6fn_sq___fw280000gn/T/ipykernel_2817/4089168952.py:2: DtypeWarning: Columns (0: somaLocation) have mixed types. Specify dtype option on import or set low_memory=False.
  mcns_meta = pd.read_csv(


,assignedOlHex1,assignedOlHex2,bodyid,flywireType,group,instance,somaSide,statusLabel,superclass,type,...,exitNerve,receptorType,somaLocation,tosomaLocation,status,top_nt,cell_type,coords,idx,sign
0,NaN,NaN,10001,DNp01,10001.0,DNp01(GF)_R,R,Roughly traced,descending_neuron,DNp01,...,NaN,NaN,[37124 22258 36274],NaN,Traced,acetylcholine,DNp01,NaN,0,1
1,NaN,NaN,10002,OCG01d,10002.0,OCG01d_L,L,Roughly traced,visual_projection,OCG01d,...,NaN,NaN,[50214 10279 27497],NaN,Traced,acetylcholine,OCG01d,NaN,1,1
2,NaN,NaN,10003,VCH,10003.0,VCH_R,R,Prelim Roughly traced,visual_centrifugal,VCH,...,NaN,NaN,[46716 21029 15714],NaN,Traced,gaba,VCH,NaN,2,-1
3,NaN,NaN,10005,AOTU019,10005.0,AOTU019_R,R,Roughly traced,cb_intrinsic,AOTU019,...,NaN,NaN,[41190 18491 12885],NaN,Traced,gaba,AOTU019,NaN,3,-1
4,NaN,NaN,10006,"VS1,VS2,VS3,VS4,VS5,VS6,VS7,VS8",10006.0,VS_L,L,Prelim Roughly traced,visual_projection,VS,...,NaN,NaN,[69604 28368 42459],NaN,Traced,acetylcholine,VS,NaN,4,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210460,NaN,NaN,1349927540,NaN,NaN,NaN,NaN,Reviewed,vnc_sensory,SNxx33,...,NaN,NaN,NaN,NaN,Traced,acetylcholine,SNxx33,NaN,161424,1
210671,NaN,NaN,1387178136,NaN,NaN,NaN,NaN,Reviewed,vnc_sensory,SNpp45,...,NaN,NaN,NaN,NaN,Traced,acetylcholine,SNpp45,NaN,161425,1
210739,NaN,NaN,1399810824,NaN,NaN,NaN,NaN,Reviewed,vnc_sensory,SNta34,...,NaN,NaN,NaN,NaN,Traced,acetylcholine,SNta34,NaN,161426,1
210918,NaN,NaN,1434599043,NaN,NaN,NaN,NaN,Reviewed,vnc_sensory,SNta31,...,NaN,NaN,NaN,NaN,Traced,acetylcholine,SNta31,NaN,161427,1


In [19]:
mcns_meta.cell_type #cell type

0           DNp01
1          OCG01d
2             VCH
3         AOTU019
4              VS
           ...   
210460     SNxx33
210671     SNpp45
210739     SNta34
210918     SNta31
211096     SNpp45
Name: cell_type, Length: 161429, dtype: str

In [20]:
mcns_meta.idx # index / neuron

0              0
1              1
2              2
3              3
4              4
           ...  
210460    161424
210671    161425
210739    161426
210918    161427
211096    161428
Name: idx, Length: 161429, dtype: int64

In [21]:
mcns_meta.bodyid # unique neuron identifier in the malecns connectome. This is what you will find in neuPrint

0              10001
1              10002
2              10003
3              10005
4              10006
             ...    
210460    1349927540
210671    1387178136
210739    1399810824
210918    1434599043
211096    1471062202
Name: bodyid, Length: 161429, dtype: int64

In [22]:
# Use type_to_function (which matched cell types with experimentally known function we loaded at the top of notebook) to map to each cell type.
mcns_meta['known_function'] = mcns_meta.cell_type.map(type_to_function).fillna('')
mcns_meta.known_function

0                                      DNp01 escape_takeoff
1                                                          
2                                                          
3         AOTU019 frontal visual field object detection;...
4                                          VS Vertical flow
                                ...                        
210460                                                     
210671                                                     
210739                                                     
210918                                                     
211096                                                     
Name: known_function, Length: 161429, dtype: str

## Building lookup dictionaries

**Why do we use lookup dictionaries?**
Our connectome data uses an ID for each individual neuron. A lookup dictionary helps us connect that ID to useful biological information, such as its cell type, side, function, or location. 

For example:
```text
neuron idx → cell type
101       → L2
102       → L2
103       → Tm1
```

This information we need is already available in the metadata table `mcns_meta` as columns, however, since most analysis functions in connectome_interpreter use dictionaries we will create them. This dictionary set-up allows `connectome_interpreter` to take connections between individual neurons and **group them into categories**. For example, instead of looking at `101 → 103`, we can summarize it as **L2 → Tm1**.

Think of a lookup dictionary as a **quick translation table: neuron ID → biological information**. 

In [33]:
mcns_idx_to_type = dict(zip(mcns_meta.idx, mcns_meta.cell_type)) # idx assigned to a type
mcns_idx_to_sign = dict(zip(mcns_meta.idx, mcns_meta.sign)) # idx assigned to a neurotransmitter sign (1: excitatory -1: inhibitory)
mcns_idx_to_function = dict(zip(mcns_meta.idx, mcns_meta.known_function)) # idx assigned to a cell type function (if known)
mcns_type_to_sign = dict(zip(mcns_meta.cell_type, mcns_meta.sign)) # cell type assigned to a neurotransmitter sign (1: excitatory -1: inhibitory)
mcns_function_type_to_sign = dict(zip(mcns_meta.known_function, mcns_meta.sign)) # cell type and function assigned to a neurotransmitter sign (1: excitatory -1: inhibitory)
mcns_idx_to_bodyid = dict(zip(mcns_meta.idx, mcns_meta.bodyid)) # idx assigned to bodyid (bodyid is a unique number to identify maleCNS neurons in neuPrint)


# Let's pick cell types to calculate effective connectivity

Here, we will work through an example of effective connectivity from the **visual system**. 

For another worked example, have a look at the [Linear Methods tutorial notebook: 1.3 Effective connectivity: source → target](https://github.com/YijieYin/connectome_interpreter) from *The Connectome Interpreter*.

## OL sensory neurons → VPN effective connectivity

Let's begin with the visual system.

For some background on how the fly visual system is organized, see [this section on the optic lobes (OL)](2-fly_brain_and_neuropils.md#optic-lobes). You can also refer to [Nern et al. (2025)](https://www.nature.com/articles/s41586-025-08746-0), which is the same dataset as the malecns connectome we are using here, but just focusing on the fully annotated right optic lobe connectome.

Before choosing our cell types, it is useful to think about what we are actually asking.

A **source** is the population of neurons whose output we want to trace. A **target** is the population of neurons that receives this information. Thus, when we calculate effective connectivity, we are asking questions such as:

> **Which target neurons receive "indirect" information from a particular source population, and how strongly are they connected?**


For this example, let's choose **LC cell types** as our target population. LC neurons are visual projection neurons (VPN) located near the end of the visual pathway, where visual information is transmitted from the optic lobes towards the brain. For our source neurons, let's choose something based on superclass: describe broad sets of neurons, grouped together by anatomy or general function (Berg et al., 2025). I will go with sensory neurons in the optic lobe for this example as I am curious how do OL sensory neurons influence VPNs despite not having direct connectivity between them.


## Choosing your own source and target populations

The examples above are just starting points. The `mcns_meta` dataframe contains several categories that can be used to define neuronal populations. For example, you might select neurons based on their `cell_type`, `superclass`, or another metadata category.

Take some time to explore the available categories and the values within them. For example:

```python
mcns_meta.columns
```

You can then inspect the unique values in a category:

```python
mcns_meta["superclass"].unique()
```

or:

```python
mcns_meta["cell_type"].unique()
```

Think about a biological question first, and then use the metadata to identify a reasonable source and target population.

A useful way to approach this is:

**Biological question → source population → target population → connectivity analysis**

For example:

```text
How does early visual information influence LC cell types, even if they are not directly connected?
            ↓
       OL Sensory Neurons
            ↓
          LC neurons
            ↓
Calculate effective connectivity
```


In [24]:
# Source: all OL sensory neurons with a known function. Target: LCs with a known function.
mcns_sensory_ol = mcns_meta.idx[(mcns_meta.superclass == 'ol_sensory') & (mcns_meta.cell_type.isin(type_to_function))] # Filtering for idx which are a part of ol_sensory superclass and are also in the type_to_function dictionary
mcns_LC = mcns_meta.idx[mcns_meta.cell_type.str.contains('LC', na=False) & (mcns_meta.cell_type.isin(type_to_function))] # Filtering for idx which are a part of LC cell types and are also in the type_to_function dictionary
mcns_LC #idx where it is a LC cell type with a known function


539          534
1289        1274
1411        1396
1595        1578
1647        1629
           ...  
168889    140189
169144    140309
190626    159095
190810    159193
193951    160247
Name: idx, Length: 3276, dtype: int64

In [25]:
# Find all 2-hop paths from OL sensory neurons to LC neurons. 
# Returns a long-format edgelist with columns: pre, post, weight, layer (1 = direct connection).
# Weight here is input proportion 
LC_sensory_ol_paths = find_paths_of_length(mcns_inprop, mcns_sensory_ol , mcns_LC, target_layer_number=2) #2 because we are interested in 2-hop connectivty
LC_sensory_ol_paths

,pre,post,weight,layer
0,117777,7,0.000076,1
1,119697,41,0.000066,1
2,38772,51,0.000259,1
3,5150,54,0.000087,1
4,22534,54,0.000437,1
...,...,...,...,...
65553,94311,160247,0.001692,2
65554,100159,160247,0.003384,2
65555,106496,160247,0.001692,2
65556,116017,160247,0.001692,2


## What happens when you don't understand how a function works?

A useful skill when working with computational tools is learning how to read the **documentation**. Documentation tells you what a function does, what information you need to give it, and what it will return.

You can find the full `connectome-interpreter` documentation [here](https://connectome-interpreter.readthedocs.io/en/latest/modules/toc.html).

For example, we are using the `find_paths_of_length()` function. You can find the documentation for this specific function [here](https://connectome-interpreter.readthedocs.io/en/latest/modules/path_finding.html#connectome_interpreter.path_finding.find_paths_of_length).

If the documentation looks confusing at first, don't worry. Documentation often uses technical terms that may be unfamiliar when you are just starting out. You can look up unfamiliar terms, ask a peer, or use AI to help you understand what the documentation is saying. The goal is not to memorize the documentation, but to learn how to **extract the information you need from it**.

### Step 1: What does the function do?

Start by looking at the first sentence of the documentation:

> “Finds the path of length `target_layer_number` between `inidx` and `outidx`...”

In simpler terms, this function looks for connections between a **source** and a **target** at a particular number of synaptic steps.

For our analysis:

* `inidx` = the **source neurons**
* `outidx` = the **target neurons**
* `target_layer_number` = how many **synaptic steps (or hops)** we want to examine

For example:

```text
Layer 1:  Source ─────────→ Target
              direct connection

Layer 2:  Source → Intermediate → Target
                    1 hop

Layer 3:  Source → Intermediate → Intermediate → Target
                    2 hops
```

The documentation tells us that **layer 1 means a direct connection from the source**. Increasing the layer number means adding more intermediates between the source and targets.

### Step 2: What information does the function need?

Next, look at the **Parameters** section of the documentation. This tells us what we need to provide to the function.

For `find_paths_of_length()`, the important parameters for us are:

| Parameter             | What it means                                   |
| --------------------- | ----------------------------------------------- |
| `edgelist`            | The connections in our network                  |
| `inidx`               | The source neurons                              |
| `outidx`              | The target neurons                              |
| `target_layer_number` | The number of synaptic steps we want to examine |

You can think of these as the **ingredients** the function needs before it can do its job.

In our example, we provide:

```python
find_paths_of_length(
    mcns_inprop,
    mcns_sensory_ol,
    mcns_LC,
    target_layer_number=2
)
```

Let's match each ingredient to the documentation:

```text
mcns_inprop       → edgelist
mcns_sensory_ol   → inidx (source neurons)
mcns_LC           → outidx (target neurons)
2                 → target_layer_number
```

### Step 3: What does the function give back?

Finally, look at the **Returns** section.

The documentation tells us that the function returns a **DataFrame containing the path data**.

This means that after running the function, we can expect a table containing information such as:

* the **pre-synaptic neuron** (`pre`);
* the **post-synaptic neuron** (`post`);
* the **layer** where that connection occurs; and
* the **weight** of the connection.

So we can store the result in a variable and inspect it:

```python
paths = find_paths_of_length(
    mcns_inprop,
    mcns_sensory_ol,
    mcns_LC,
    target_layer_number=2
)

paths.head()
```

When you encounter a new function, try asking yourself these three questions:

**1. What does it do?**
→ Look at the description.

**2. What does it need from me?**
→ Look at the parameters.

**3. What will I get back?**
→ Look at the returns section.

For the future functions we use, try to work through the documentation to understand what are the parameters and returns of the function along with the tutorial to gain a better understading of how to use the toolkit!

Since our connections are between neurons in OL sensory and LCs, we want to group them into cell type categories. But not just that, I want to go a step futher and group them by cell type and also function. Recall we made a dictionary exactly for this purpose early on? It's `mcns_idx_to_function`. So let's use that with a grouping function in he Connectome Interpreter `group_path`. Have a read of the documentation for yourself. What are the parameters. What will it return?


In [26]:
# Since the previous function called the 2-hop connections between Ol sensory neurons and LCs.
# It is time to add the cell type annotation to these in preparation to visualise these connections
#  Group paths by cell type and function
LC_sensory_ol_paths_func = group_paths(LC_sensory_ol_paths, pre_group=mcns_idx_to_function, post_group=mcns_idx_to_function)
LC_sensory_ol_paths_func

,layer,pre,post,weight
0,1,HBeyelet circadian photosensitivity,,0.000001
1,1,HBeyelet circadian photosensitivity,DNp27 neuromodulatory,0.000129
2,1,HBeyelet circadian photosensitivity,Tm5c Green detection,0.000082
3,1,HBeyelet circadian photosensitivity,aMe3 input to clock neurons,0.119512
4,1,R7d polarized UV detection,,0.000005
...,...,...,...,...
377,2,aMe3 input to clock neurons,"LC26 visual_loom, visual_small_object",0.000043
378,2,aMe3 input to clock neurons,LC4 Looming detector; visual_loom; flight; esc...,0.000004
379,2,aMe3 input to clock neurons,"LC9 visual_loom, visual_small_object; courtshi...",0.000033
380,2,aMe3 input to clock neurons,"LPLC1 visual_loom, collision avoidance",0.000028


After grouping we can finally approach effective connectivity. Remmeber our `LC_sensory_ol_paths_func` dataframe are direct connections with everything from source as OL sensory neurons and LC cell types as targets. After all, that is why we are calculating effective connectivity. As these two groups of neurons are not necesarily synaptically connected. Rather, there are many intermediate players that connect in between which is why you are seeing different cell types than LCs in the post column and different cells than OL sensory neurons in the pre column. We will go into more details in the next tutorial about how this calculation is done, but for now this basic understanding is important. 

Since this is not effective connectivity, but rather direct synaptic connection from everything in between OL sensory -> X -> LC, we need a function that calculates the effective connectivity from this dataframe. That is where `effective_conn_from_paths` comes in. Once again, look at the documentation for the ingredients (aka parameters) you need to feed the function


In [27]:
# Calculate and visualise effective connectivity from each OL sensory cell type and LC cell type.
eff = effective_conn_from_paths(LC_sensory_ol_paths_func) 
display_df(eff.iloc[:20, :20]) # display 20 rows and columns 

GPU not available, using CPU instead.


post,LC10a visual_small_object; courtship; visual,LC10b visual_small_object,LC10d visual_small_object,LC10e visual_small_object,LC11 Small object; visual_small_object; visual,"LC12 visual_loom, visual_large_objects, visual_small_object; visual","LC15 visual_thin_vertical_bar, visual_large_objects; visual","LC16 Retreat behavior; visual_loom; backwards, freezing; walking","LC17 visual_loom, visual_large_objects, visual_small_object; electrical synapses for object tracking in flight",LC18 small object detection,LC21 visual_small_object,LC25 complex line motion; visual_thin_vertical_bar,"LC26 visual_loom, visual_small_object",LC4 Looming detector; visual_loom; flight; escape_takeoff,"LC6 visual_loom, visual_small_object; flight; escape_takeoff","LC9 visual_loom, visual_small_object; courtship; backward walking","LPLC1 visual_loom, collision avoidance",LPLC2 Looming escape
pre,,,,,,,,,,,,,,,,,,
HBeyelet circadian photosensitivity,0.000000,0.000001,0.000001,0.000000,0.000000,0.000011,0.000000,0.000004,0.000002,0.000000,0.000000,0.000001,0.000005,0.000001,0.000000,0.000004,0.000004,0.000003
R7d polarized UV detection,0.000005,0.000005,0.000011,0.000006,0.000001,0.000002,0.000000,0.000001,0.000001,0.000001,0.000001,0.000003,0.000002,0.000001,0.000008,0.000005,0.000001,0.000002
R7p UV detection,0.000105,0.000248,0.000054,0.000044,0.000016,0.000014,0.000018,0.000042,0.000195,0.000013,0.000029,0.000246,0.000317,0.000015,0.001676,0.000060,0.000053,0.000115
R7y UV detection,0.000026,0.000058,0.000045,0.000034,0.000019,0.000012,0.000020,0.000054,0.000015,0.000010,0.000027,0.000068,0.000106,0.000011,0.000123,0.000017,0.000028,0.000039
R8d polarized UV detection,0.000001,0.000008,0.000014,0.000004,0.000001,0.000009,0.000002,0.000002,0.000001,0.000001,0.000001,0.000016,0.000002,0.000002,0.000006,0.000000,0.000003,0.000009
R8p Blue detection,0.000017,0.000075,0.000101,0.000032,0.000013,0.000067,0.000019,0.000042,0.000014,0.000010,0.000019,0.000143,0.000056,0.000021,0.000111,0.000006,0.000034,0.000077
R8y green detection,0.000058,0.000330,0.000472,0.000138,0.000022,0.000318,0.000051,0.000077,0.000066,0.000019,0.000039,0.000612,0.000139,0.000080,0.000589,0.000026,0.000108,0.000317


# What am I looking at?
As you can see, effective connectivity from the photoreceptor level to the LC (VPN) level is generally weak. However, R7p, a UV-sensitive photoreceptor, shows noticeably stronger influence on LC6, a visual looming-related cell type.

This raises an interesting question: why might the R7p → LC6 pathway stand out from the other photoreceptor → LC connections? You could use this as starting point to explore the literature and investigate what is known about R7p, LC6, and their roles in visual processing. You may also find that examining the intermediate cell types in this pathway gives you additional clues and questions to proceed on futher.


# Signed effective connectivity

Effective connectivity tells us whether two cell types can influence one another through **indirect paths**, rather than only through a direct synaptic connection. Here, we will also keep track of whether those paths are associated with **excitatory or inhibitory influence**.

Before calculating signed effective connectivity, we will group our paths by **cell types and function again**.

Although this tutorial always groups by cell type and function, this does not mean that we should always group by these variables. Depending on the question you are asking, it may be more appropriate to work at the individual-neuron level using **body IDs**. For example, if you are interested in a particular neuron's signed effective connectivity and whether it differs in a cell type, keeping the body IDs would preserve that information.

The `signed_effective_conn_from_paths()` function can work with any grouping you choose. Before the function, you need to map idx -> grouping of choice (if any) as just explained. Then for `signed_effective_conn_from_paths()` we need to map that grouping you chose to the NT sign (eg., cell type -> NT sign, bodyid -> NT sign, and etc). In our `mcns_meta` dataframe, NT sign information is stored in the `"sign"` column, where `+1` represents an excitatory sign and `-1` represents an inhibitory sign according to the convention used in this dataset.

The function returns two dataframes representing the effective connectivity between sources and targets, one for effective excitation, the other inhibition. The neurotransmitter/sign annotation of the allows us to separate these paths into excitatory and inhibitory components. For more details [see](https://connectome-interpreter.readthedocs.io/en/latest/tutorials/ei_matmul.html)..

`eff_exc` contains the effective connectivity associated with **excitatory** paths, while `eff_inh` contains the effective connectivity associated with **inhibitory** paths.



In [ ]:
# group by type and function
LC_sensory_ol_paths_type = group_paths(LC_sensory_ol_paths, pre_group=mcns_idx_to_function, post_group=mcns_idx_to_function)
LC_sensory_ol_paths_type

,layer,pre,post,weight
0,1,HBeyelet circadian photosensitivity,,0.000001
1,1,HBeyelet circadian photosensitivity,DNp27 neuromodulatory,0.000129
2,1,HBeyelet circadian photosensitivity,Tm5c Green detection,0.000082
3,1,HBeyelet circadian photosensitivity,aMe3 input to clock neurons,0.119512
4,1,R7d polarized UV detection,,0.000005
...,...,...,...,...
377,2,aMe3 input to clock neurons,"LC26 visual_loom, visual_small_object",0.000043
378,2,aMe3 input to clock neurons,LC4 Looming detector; visual_loom; flight; esc...,0.000004
379,2,aMe3 input to clock neurons,"LC9 visual_loom, visual_small_object; courtshi...",0.000033
380,2,aMe3 input to clock neurons,"LPLC1 visual_loom, collision avoidance",0.000028


In [37]:
# Find signed effective connectivity between source and target
eff_exc, eff_inh = signed_effective_conn_from_paths(LC_sensory_ol_paths_type, idx_to_nt=mcns_function_type_to_sign)
eff_exc

post,LC10a visual_small_object; courtship; visual,LC10b visual_small_object,LC10d visual_small_object,LC10e visual_small_object,LC11 Small object; visual_small_object; visual,"LC12 visual_loom, visual_large_objects, visual_small_object; visual","LC15 visual_thin_vertical_bar, visual_large_objects; visual","LC16 Retreat behavior; visual_loom; backwards, freezing; walking","LC17 visual_loom, visual_large_objects, visual_small_object; electrical synapses for object tracking in flight",LC18 small object detection,LC21 visual_small_object,LC25 complex line motion; visual_thin_vertical_bar,"LC26 visual_loom, visual_small_object",LC4 Looming detector; visual_loom; flight; escape_takeoff,"LC6 visual_loom, visual_small_object; flight; escape_takeoff","LC9 visual_loom, visual_small_object; courtship; backward walking","LPLC1 visual_loom, collision avoidance",LPLC2 Looming escape
pre,,,,,,,,,,,,,,,,,,
HBeyelet circadian photosensitivity,6.100364e-08,7.164105e-08,3.598563e-08,2.722796e-08,5.058615e-08,2.057398e-08,7.698576e-08,1.940984e-07,2.847239e-08,3.341324e-08,7.261080e-08,1.385028e-07,2.107955e-07,2.545693e-08,6.328146e-08,2.837743e-08,1.057698e-07,9.679345e-08
R7d polarized UV detection,5.347159e-06,3.870211e-06,9.370858e-06,5.268264e-06,5.348570e-07,4.187383e-07,2.500938e-07,9.336243e-07,9.834824e-07,1.093765e-06,5.569490e-07,1.325875e-06,1.627495e-06,7.395733e-07,7.554409e-06,5.044044e-06,7.043241e-07,1.015762e-06
R7p UV detection,1.035300e-04,2.047717e-04,3.293396e-05,3.585484e-05,1.558694e-05,5.402787e-06,1.414614e-05,4.046563e-05,1.947333e-04,1.289417e-05,2.404760e-05,2.228544e-04,2.413046e-04,1.271362e-05,1.669583e-03,5.924472e-05,4.869303e-05,1.042799e-04
R7y UV detection,2.534192e-05,3.202880e-05,2.667465e-05,2.774287e-05,1.864095e-05,2.674388e-06,1.763123e-05,5.311864e-05,1.476550e-05,9.322384e-06,2.398705e-05,4.832650e-05,6.826672e-05,8.794228e-06,1.171766e-04,1.630217e-05,2.469479e-05,2.987202e-05
R8d polarized UV detection,3.166203e-07,4.743577e-07,1.759913e-07,1.317949e-07,5.509105e-07,1.594531e-07,4.292974e-07,1.029483e-06,2.448875e-07,5.219566e-07,6.472172e-07,8.788985e-07,1.246197e-06,3.419384e-07,1.414186e-06,8.510774e-08,5.834835e-07,5.869978e-07
R8p Blue detection,1.300573e-05,2.108628e-05,6.624613e-06,5.127874e-06,1.187888e-05,2.627150e-06,1.106213e-05,3.735443e-05,1.197539e-05,7.903388e-06,1.442593e-05,3.680902e-05,5.075473e-05,6.394300e-06,8.021360e-05,3.878743e-06,1.817852e-05,2.164779e-05
R8y green detection,3.581163e-05,6.802342e-05,1.080145e-05,8.436273e-06,1.433069e-05,4.029117e-06,1.417926e-05,5.479696e-05,5.530361e-05,9.923348e-06,1.807839e-05,9.325030e-05,1.146242e-04,8.939870e-06,4.411072e-04,1.433113e-05,3.186410e-05,4.802506e-05


In [38]:
eff_inh

post,LC10a visual_small_object; courtship; visual,LC10b visual_small_object,LC10d visual_small_object,LC10e visual_small_object,LC11 Small object; visual_small_object; visual,"LC12 visual_loom, visual_large_objects, visual_small_object; visual","LC15 visual_thin_vertical_bar, visual_large_objects; visual","LC16 Retreat behavior; visual_loom; backwards, freezing; walking","LC17 visual_loom, visual_large_objects, visual_small_object; electrical synapses for object tracking in flight",LC18 small object detection,LC21 visual_small_object,LC25 complex line motion; visual_thin_vertical_bar,"LC26 visual_loom, visual_small_object",LC4 Looming detector; visual_loom; flight; escape_takeoff,"LC6 visual_loom, visual_small_object; flight; escape_takeoff","LC9 visual_loom, visual_small_object; courtship; backward walking","LPLC1 visual_loom, collision avoidance",LPLC2 Looming escape
pre,,,,,,,,,,,,,,,,,,
HBeyelet circadian photosensitivity,5.511578e-08,6.637645e-07,0.000001,3.281822e-07,3.694260e-07,0.000011,9.287770e-08,3.819191e-06,1.740363e-06,2.150433e-08,5.208157e-08,0.000001,5.217837e-06,6.721918e-07,3.747899e-07,4.009144e-06,3.575925e-06,0.000003
R7d polarized UV detection,8.555595e-08,1.279173e-06,0.000002,5.177964e-07,3.215781e-08,0.000001,1.629182e-07,9.170768e-08,4.175550e-08,3.297005e-08,1.133757e-07,0.000002,7.344670e-07,2.604842e-07,5.656775e-07,4.497192e-08,2.932106e-07,0.000001
R7p UV detection,1.235201e-06,4.351862e-05,0.000021,8.323530e-06,6.302831e-07,0.000009,4.239982e-06,1.824541e-06,5.082881e-07,4.397241e-07,4.917996e-06,0.000023,7.523069e-05,1.954983e-06,6.545340e-06,6.482534e-07,3.852189e-06,0.000011
R7y UV detection,9.671585e-07,2.608526e-05,0.000018,6.249420e-06,4.351261e-07,0.000009,2.718053e-06,1.205719e-06,3.993981e-07,3.579956e-07,2.806803e-06,0.000020,3.821282e-05,2.097627e-06,5.642205e-06,4.419838e-07,3.080834e-06,0.000010
R8d polarized UV detection,6.446652e-07,7.763763e-06,0.000014,3.838604e-06,2.309095e-07,0.000009,1.086350e-06,6.637239e-07,3.263390e-07,2.515268e-07,6.091754e-07,0.000015,7.125229e-07,2.099090e-06,4.383754e-06,3.498669e-07,2.246994e-06,0.000008
R8p Blue detection,4.458906e-06,5.374957e-05,0.000094,2.661800e-05,1.612578e-06,0.000064,7.524885e-06,4.631506e-06,2.276203e-06,1.794527e-06,4.285127e-06,0.000106,5.273782e-06,1.450079e-05,3.031880e-05,2.487168e-06,1.557547e-05,0.000055
R8y green detection,2.179075e-05,2.622395e-04,0.000461,1.300406e-04,7.869256e-06,0.000314,3.673845e-05,2.258426e-05,1.110623e-05,8.747867e-06,2.085757e-05,0.000519,2.452619e-05,7.089965e-05,1.481962e-04,1.210078e-05,7.608755e-05,0.000269


The two tables have the same source and target cell types, but the values represent different signed components of the effective connectivity.

For example, if an OL sensory neuron can reach a VPN through an excitatory intermediate neuron, that path contributes to the corresponding entry in `eff_exc`. If it reaches the same VPN through an inhibitory intermediate neuron, that path contributes to `eff_inh`.

## Exploring the signed connectivity

You can now use the two matrices to ask questions such as:

1. **Which OL sensory cell types have the strongest effective excitatory influence on each LC VPN cell type?**

2. **Which OL sensory cell types have the strongest effective inhibitory influence on each LC VPN cell type?**

3. **Do some OL sensory cell types have both excitatory and inhibitory routes to the same LC VPN cell type?**

4. **Are particular LC VPN cell types predominantly reached through excitatory or inhibitory paths?**

## Important caveat: neurotransmitter predictions and biological sign

The excitatory/inhibitory signs used here come from **predicted neurotransmitter annotations**, rather than being manually assigned from the literature for every cell type. The Connectome Interpreter uses the neurotransmitter predictions from [Eckstein et al., 2024](https://www.sciencedirect.com/science/article/pii/S0092867424003076?via%3Dihub) also see [here](https://connectome-interpreter.readthedocs.io/en/latest/tutorials/ei_matmul.html) to estimate the sign of effective connectivity.

This creates an important caveat for certain cell types, such as some visual system cell types. Drosophila photoreceptors (OL sensory cell types) are known from experimental studies to release **histamine**, which acts as an inhibitory neurotransmitter in their downstream visual pathways. In this review [Currier et al., 2023](https://academic.oup.com/genetics/article/224/2/iyad064/7147603). See section: **Optic Lobes**, then sub-section: **Photoreceptors feed into circuits that process motion, color, and polarization via histaminergic transmission.** In the first paragraph, they note histamine acts as an inhibtiory neurotrasmitter to downstream connections. However, histamine was not one of the six neurotransmitters predicted by the model used in Eckstein et al. (2024).

Therefore, a photoreceptor receiving a `+1` sign in this dataset does **not automatically mean it is biologically excitatory**.

This is a useful example of why connectomic analyses should be interpreted alongside the biological literature, particularly because, as you have learned, connectome datasets and their annotations have important limitations. When a computational annotation appears inconsistent with established biology, ask where the annotation came from, what assumptions were made, and what limitations the underlying prediction model has. Importantly, what you should takeaway is synapse-level connectomes don't contain all the information and the whole field is working on this to add more "annotations" to the connnectome. 

However, what connectomics does allow us to investigate is questions that have not yet been described in the literature. The resolution of synapse-level connectomes provides detailed maps of neuronal connectivity that were not available when many previous studies were published. Connectivity can therefore reveal new relationships and generate hypotheses about how information may flow through a circuit. However, connectivity alone generally cannot tell us the functional properties of a neuron, such as its neurotransmitter identity or whether its effect on a downstream neuron is excitatory or inhibitory. For these properties, it is important to examine the existing literature and determine what is already known. From this we can conclude perhaps, it's not appropriate then to use the sign property for our example question. 

See below from the mcns_meta dataframe the sign of the photoreceptors. You will notice they are +1 under column "sign".

In [39]:
mcns_meta[mcns_meta.superclass == 'ol_sensory']

,assignedOlHex1,assignedOlHex2,bodyid,flywireType,group,instance,somaSide,statusLabel,superclass,type,...,receptorType,somaLocation,tosomaLocation,status,top_nt,cell_type,coords,idx,sign,known_function
1065,NaN,NaN,11139,R1-6,11139.0,R1-R6_L,NaN,Reviewed,ol_sensory,R1-R6,...,NaN,NaN,NaN,Traced,histamine,R1-R6,NaN,1051,1,R1-R6 Achromatic vision
4162,NaN,NaN,14428,HBeyelet,14428.0,HBeyelet_L,L,Roughly traced,ol_sensory,HBeyelet,...,NaN,NaN,NaN,Traced,histamine,HBeyelet,NaN,4093,1,HBeyelet circadian photosensitivity
5122,NaN,NaN,15479,R8,15479.0,R8p_L,L,Reviewed,ol_sensory,R8p,...,NaN,[86291 16900 23873],NaN,Traced,histamine,R8p,NaN,5023,1,R8p Blue detection
5257,NaN,NaN,15625,R8,15625.0,R8y_L,L,Reviewed,ol_sensory,R8y,...,NaN,[89018 16908 26868],NaN,Traced,histamine,R8y,NaN,5150,1,R8y green detection
5929,NaN,NaN,16357,R1-6,11139.0,R1-R6_L,NaN,Out of scope,ol_sensory,R1-R6,...,NaN,NaN,NaN,NaN,histamine,R1-R6,NaN,5804,1,R1-R6 Achromatic vision
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
205145,NaN,NaN,983968288,R8,16681.0,R8d_R,NaN,Prelim Roughly traced,ol_sensory,R8d,...,NaN,NaN,NaN,Traced,histamine,R8d,NaN,161004,1,R8d polarized UV detection
205338,NaN,NaN,997019444,R7,20107.0,R7p_L,NaN,Reviewed,ol_sensory,R7p,...,NaN,NaN,NaN,Traced,histamine,R7p,NaN,161015,1,R7p UV detection
205376,NaN,NaN,1000116295,R8,15625.0,R8y_L,NaN,Reviewed,ol_sensory,R8y,...,NaN,NaN,NaN,Traced,histamine,R8y,NaN,161018,1,R8y green detection
205677,NaN,NaN,1019617211,R7,26406.0,R7_unclear_R,NaN,RT Hard to trace,ol_sensory,R7_unclear,...,NaN,NaN,NaN,Traced,histamine,R7_unclear,NaN,161051,1,


## How do we calculate the total signed effective connectivity onto a target neuron?

Here, I will use the biological example of **OL sensory cell types → 2-hop effective connectivity → LC cell types**. We will ask:

> **Are particular LC VPN cell types predominantly reached through excitatory or inhibitory paths?**

To answer this, we can calculate the **net effective connectivity** by comparing the excitatory and inhibitory components. In other words, for each OL sensory cell type → LC cell type pair, we can ask whether the effective connectivity is more strongly associated with excitation or inhibition.

To see an example of this approach being used in a mushroom body circuit, along with the biological logic behind the analysis, see the Connectome Interpreter tutorial under [Showcase: appetitive vs aversive MBON drive onto DNs](https://colab.research.google.com/drive/145Td8_fFTPwTsDdEkQGAgdZ7CFQ8qXnr#scrollTo=pXpKvtJ2HZl9).

**Important:** for the purposes of this example, I will still use the sign annotations in our dataset to demonstrate how the calculation works. However, as we discussed above, the sign annotation for photoreceptors has an important limitation because of their histaminergic neurotransmission. Therefore, we should not use the resulting excitatory/inhibitory values as a direct biological interpretation of photoreceptor influence.


In [40]:
# Net influence
net = eff_exc.fillna(0) - eff_inh.fillna(0)
net_lc = net.sum(axis=0).reset_index()
net_lc.columns = ['LC', 'net']
net_lc

,LC,net
0,LC10a visual_small_object; courtship; visual,0.000154
1,LC10b visual_small_object,-0.000065
2,LC10d visual_small_object,-0.000525
3,LC10e visual_small_object,-0.000093
4,LC11 Small object; visual_small_object; visual,0.000050
5,"LC12 visual_loom, visual_large_objects, visual...",-0.000402
6,"LC15 visual_thin_vertical_bar, visual_large_ob...",0.000005
7,"LC16 Retreat behavior; visual_loom; backwards,...",0.000153
8,"LC17 visual_loom, visual_large_objects, visual...",0.000262
9,LC18 small object detection,0.000030
